# NepalDTM-G: 1 m generation for one province on Colab

Set `PROVINCE` below and run all cells. Each finished 10 km tile is uploaded to the R2 bucket `nepaldtm-g-1m` at
`<province>/tiles/`; tiles already there are skipped, so after any disconnect just run all cells again.

Provinces: `Koshi`, `Madhesh`, `Gandaki`, `Lumbini`, `Karnali`, `Sudurpashchim` (Bagmati is done).

**Before you start:** Runtime → Change runtime type → **A100 GPU**, High-RAM on. Colab secrets (key icon, notebook
access on): `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_ENDPOINT` (token with Object Read & Write on `nepaldtm-g-1m`).

In [ ]:
PROVINCE = "Koshi"  # <- change this

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import userdata
for k in ("R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_ENDPOINT"):
    os.environ[k] = userdata.get(k)
os.environ["R2_BUCKET"] = "nepaldtm-g-1m"
P = PROVINCE.lower()
print("secrets loaded;", PROVINCE)

In [ ]:
!pip -q install rasterio shapely pyproj boto3 affine
!rm -rf /content/NepalDTM && git clone -q --depth 1 -b g https://github.com/achyut/NepalDTM /content/NepalDTM

Inputs: 8 m NepalDTM v2 for the province on the national grid, the province boundary and the model weights.

In [ ]:
B = "https://nepaldtm-data.achyutpdl.com/g/colab"
!mkdir -p /content/in
!cd /content/in && curl -sSfL -A curl/8 -C - -O {B}/{P}/{P}_v2_dtm_8m.tif -O {B}/{P}/{P}.geojson -O {B}/{P}/SHA256SUMS
!cd /content/in && [ -f ema_150000.pt ] || curl -sSfL -A curl/8 -O {B}/ema_150000.pt
!cd /content/in && sha256sum -c SHA256SUMS

## Benchmark (optional)
Picks the largest batch that fits the GPU. Skip it to use 32, which fits a 40 GB A100.

In [ ]:
import sys, time, numpy as np, torch, rasterio
sys.path.insert(0, "/content/NepalDTM/code/g")
from g_model import DEV, UNet, sample
net = UNet().to(DEV); net.load_state_dict(torch.load("/content/in/ema_150000.pt", map_location=DEV)["ema"]); net.eval()
src = rasterio.open(f"/content/in/{P}_v2_dtm_8m.tif")
z = src.read(1, window=((src.height // 2, src.height // 2 + 320), (src.width // 2, src.width // 2 + 320))).astype("float32")
z = np.nan_to_num(z, nan=float(np.nanmean(z)))
wins = torch.from_numpy(np.stack([z[i:i + 32, j:j + 32] for i in range(0, 288, 24) for j in range(0, 288, 24)]))[:, None]
best = None
for bs in (8, 32, 64, 128):
    x = wins[:bs].to(DEV).repeat_interleave(2, 0)
    try:
        with torch.no_grad():
            sample(net, x, n_steps=10); torch.cuda.synchronize(); t = time.time()
            for _ in range(3): sample(net, x, n_steps=10)
            torch.cuda.synchronize()
    except torch.OutOfMemoryError:
        print(f"batch {bs:4d}: out of GPU memory"); del x; torch.cuda.empty_cache(); break
    spw = (time.time() - t) / 3 / bs
    best = min(best or (9, 0), (spw, bs))
    print(f"batch {bs:4d}: {spw:.3f} s/window  ({0.53 / spw:.1f}x the laptop)")
BS = best[1]; print("use --bs", BS)
for v in ("net", "x", "wins"):
    globals().pop(v, None)
import gc; gc.collect(); torch.cuda.empty_cache()

## Generate
Re-run all cells after a disconnect; finished tiles are skipped. It prints `done` when the province is complete.

In [ ]:
BS = globals().get('BS', 32)
!cd /content/NepalDTM/code && python -u g/g_region.py --region {PROVINCE} \
    --dtm /content/in/{P}_v2_dtm_8m.tif --prov /content/in/{P}.geojson --ckpt /content/in/ema_150000.pt \
    --steps 10 --ens 2 --bs {BS} --out /content/out --r2-prefix {P}/tiles
!rm -f /content/out/tiles/*.tif  # uploaded; keep the Colab disk free